In [ ]:
# Import required libraries
import os
import kagglehub
from torch.utils.data import Dataset, DataLoader , random_split
from torchvision import transforms
import matplotlib.pyplot as plt
import os
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from PIL import Image
from glob import glob
import torch.nn as nn
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import torch.optim as optim
import pandas as pd
from torchvision import models
from torchvision.datasets import ImageFolder
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from sklearn.model_selection import train_test_split
from torchvision.datasets import ImageFolder





In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:

from glob import glob

class PetDataset(Dataset):
    def __init__(self, image_paths, img_mask, split,target_types, transform=None, target_transform =None):
        # self.image_paths = glob(os.path.join(image_paths, split, "*.jpg"))
        # self.img_mask= glob(os.path.join(img_mask, target_types, "*.png"))
        self.image_paths=image_paths
        self.img_mask=img_mask
        self.split = split
        self.target_types=target_types
        self.transform = transform
        self.target_transform= target_transform


        self.image_paths = sorted(
        glob(os.path.join(self.image_paths, split, "*.jpg")))

        self.img_mask = sorted(
        glob(os.path.join(self.img_mask, target_types, "*.png")))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.img_mask[idx]

        image = Image.open(path).convert("RGB")
        label = Image.open(label).convert("L")


        if self.transform:
            image = self.transform(image)
            label =  self.target_transform(label)


        return image, label

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader
from torch import nn

# Define transforms for images
img_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# need to subtract 1 from the make to bring it from 1-3 range to 0-2 range (CrossEntropyLoss requires labels to start from 0)
class SubtractOne(nn.Module):
  def forward(self, img):
    return img-1

# Define transforms for masks

target_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),                     # Convert PIL -> tensor
    # transforms.Lambda(lambda t: t.squeeze(0).long()),  # Remove channel, convert to long
    # transforms.Lambda(lambda t: t.clamp(max=4)) # Replace 255 with 4 (or 0 if you prefer
    # mask = mask.float() / 255  # normalize to 0-1 )
])                                          ## Question: What do you think would happen if we
                                                            ##  added rotation augmentation to the image only?


path ="/kaggle/input/q3-stage3-2026/dataset"

full_dataset= train_dataset = PetDataset(
   path , path , split="images", target_types="masks",
  transform=img_transforms, target_transform=target_transforms
)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])


train_dataset = PetDataset(
     path, path , split="images", target_types="masks",
    transform=img_transforms, target_transform=target_transforms
)

test_dataset = PetDataset(
    path, path , split="images", target_types="masks",
    transform=img_transforms, target_transform=target_transforms
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Train Dataset: {len(train_dataset)} images")
print(f"Test Dataset: {len(test_dataset)} images")




In [ ]:
# TO DO

# TO DO
#!pip install segmentation_models_pytorch
import segmentation_models_pytorch as smp

# Define U-Net Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = smp.Unet(
    encoder_name="efficientnet-b0",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  # Binary segmentation (5 output channel)
      # Apply Sigmoid activation directly in the model
    # criterion = torch.nn.BCELoss() for "sigmoid" # <--- Use BCELoss, NOT BCEWithLogitsLoss
).to(device)


In [ ]:
# TO DO
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)  # mask shape becomes [N, H, W]

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)    # mask shape becomes [N, H, W]

            outputs = model(images)  # Now [N, H, W]
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO

import torch
from torch import nn
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
# TO DO
# TO DO
import random
import matplotlib.pyplot as plt

model.eval()
# Get some test samples
test_samples = random.sample(range(len(test_dataset)), 3)

for idx in test_samples:
    img, mask = test_dataset[idx]

    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device))  # Forward pass

    pred_mask = torch.softmax(pred_mask, dim=1)  # Convert logits to probabilities
    pred_mask = pred_mask.argmax(dim=1).cpu().squeeze().numpy()  # Get class with highest probability

    # Display images
    fig, axes = plt.subplots(1, 3, figsize=(10, 5))
    axes[0].imshow(img.permute(1, 2, 0))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    axes[2].imshow(pred_mask, cmap="gray")  # Show class map
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()
